# Olist Customer Growth Intelligence — v2.2: three attempts to make "will this customer come back?" stronger

**Run on Kaggle:** *Add Input* → Olist dataset → **Save Version → Save & Run All**.

v2.1 (gradient boosting, 43 features) reached a gapped out-of-time AUC of **0.557** on the real data, learning from only
65 repeat buyers. v2.2 tests three ideas, separately and together, on **the same test customers** (scored from 2017-10-01,
with a full 180 days of follow-up):

1. **Survival (time-to-return) model.** v2.1 can only train on customers with a complete 180-day follow-up before the
   test period, so it throws most customers away. A discrete-time survival model also learns from *partial* follow-up
   ("no return in the first 60 days so far"). Training data is cut off at the test start, so nothing from the test period leaks in.
2. **Category & product return rates.** How often earlier customers of the same category / product came back, computed only
   from data before the test period, shrunk toward the average, and out-of-fold for training customers.
3. **Review-text sentiment.** A text model learned from Olist's own pre-test reviews (comment → stars). No external model,
   no internet needed.

**Rule fixed in advance:** a v2.2 model replaces v2.1 only if the lower end of its 95% AUC interval is above v2.1's AUC
on the same test. Five candidates are tested, so a narrow pass should be read with that in mind.

In [4]:
import os, json, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
try:
    display
except NameError:          # running outside Jupyter
    display = print

SEED = 42
WINDOW_DAYS, MIN_GAP_DAYS = 180, 1.0            # same repeat definition as v1/v2 (used in section 4)
RESOLVE_DAYS = 60                                # an order's outcome is treated as known 60 days after purchase
TEST_START = pd.Timestamp("2018-03-01")          # out-of-time test period starts here
TRAIN_END_GAPPED = TEST_START - pd.Timedelta(days=RESOLVE_DAYS)
TARGETS = ["went_wrong", "late", "bad_review", "canceled"]
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    cands = [Path(os.environ["OLIST_DIR"])] if os.environ.get("OLIST_DIR") else []
    for root, _, files in os.walk("/kaggle/input"):
        if "olist_orders_dataset.csv" in files:
            cands.append(Path(root)); break
    cands.append(Path("."))
    for c in cands:
        if (c / "olist_orders_dataset.csv").exists():
            return c
    raise FileNotFoundError("Attach the Olist dataset as a notebook input (or set OLIST_DIR).")

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR, "| Output directory:", OUT_DIR)

def wilson(k, n, z=1.959964):
    """Wilson 95% interval in percent (no statsmodels needed)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))

def rate_table(df, by, target):
    t = df.groupby(by, observed=True)[target].agg(orders="size", events="sum").reset_index()
    t["rate_pct"] = (t.events / t.orders * 100).round(2)
    ci = [wilson(k, n) for k, n in zip(t.events, t.orders)]
    t["ci_low"] = [round(c[0], 2) for c in ci]; t["ci_high"] = [round(c[1], 2) for c in ci]
    return t.rename(columns={by: "band"})

def attach(left, right, on="order_id"):
    """Left-join that refuses overlapping columns and row multiplication (grain guard)."""
    dup = [c for c in right.columns if c in left.columns and c != on]
    assert not dup, f"overlapping columns {dup}"
    out = left.merge(right, on=on, how="left", validate="many_to_one")
    assert len(out) == len(left), "join changed row count"
    return out

Data directory: /kaggle/input/datasets/olistbr/brazilian-ecommerce | Output directory: /kaggle/working


In [5]:
# v2.1 uses v2's test period (customers scored from Oct 2017 on); seller-history global rates use only data
# known before the strictest training cut-off (test start - 180 days).
TEST_START = pd.Timestamp("2017-10-01")
TRAIN_END_GAPPED = TEST_START - pd.Timedelta(days=180)
WINDOWS = [180, 90]

In [6]:
rd = lambda name: pd.read_csv(DATA_DIR / name)
customers, orders = rd("olist_customers_dataset.csv"), rd("olist_orders_dataset.csv")
items, pay, reviews = rd("olist_order_items_dataset.csv"), rd("olist_order_payments_dataset.csv"), rd("olist_order_reviews_dataset.csv")
products, sellers, geo = rd("olist_products_dataset.csv"), rd("olist_sellers_dataset.csv"), rd("olist_geolocation_dataset.csv")
try:
    trans = rd("product_category_name_translation.csv")
except FileNotFoundError:
    trans = None

for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])

orders = orders.merge(customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]],
                      on="customer_id", how="left", validate="many_to_one")
assert orders.order_id.is_unique and orders.customer_unique_id.notna().all()
orders["is_value_order"] = ~orders.order_status.isin(["canceled", "unavailable"])
OBS_END = orders.loc[orders.is_value_order, "order_purchase_timestamp"].max()   # same primary definition as v2
print(f"Orders: {len(orders):,} | customers: {orders.customer_unique_id.nunique():,} | sellers: {len(sellers):,} | "
      f"geolocation rows: {len(geo):,} | observation end (last value order): {OBS_END}")

Orders: 99,441 | customers: 96,096 | sellers: 3,095 | geolocation rows: 1,000,163 | observation end (last value order): 2018-09-03 09:06:57


## 1. Data preparation (identical to v2.1 and v3)

In [7]:
o = orders.copy()
o["canceled"] = (~o.is_value_order).astype(int)
deliv, est = o.order_delivered_customer_date, o.order_estimated_delivery_date
o["late"] = np.where(o.canceled == 1, np.nan,                                   # not defined for canceled orders
            np.where(deliv.notna(), (deliv.dt.normalize() > est).astype(float),  # delivered after the promised day
            np.where((est < OBS_END) & (o.order_status != "delivered"), 1.0, np.nan)))  # never delivered and promise passed

rv = reviews.groupby("order_id").agg(min_review=("review_score", "min"), review_answered=("review_answer_timestamp", "max")).reset_index()
o = attach(o, rv)
o["has_review"] = o.min_review.notna().astype(int)
o["bad_review"] = (o.min_review <= 2).astype(int)
o["went_wrong"] = ((o.canceled == 1) | (o.late == 1) | (o.bad_review == 1)).astype(int)

n_items_per_order = items.groupby("order_id").size()
o["has_items"] = o.order_id.isin(n_items_per_order.index)
o["resolved"] = o.order_purchase_timestamp <= OBS_END - pd.Timedelta(days=RESOLVE_DAYS)
o["in_population"] = o.resolved & o.has_items & (o.late.notna() | (o.canceled == 1))

no_items = o[~o.has_items]
print(f"Orders with NO item rows: {len(no_items):,}, of which canceled/unavailable: {no_items.canceled.mean()*100:.1f}%.")
print("  -> excluded: at purchase time every order has items, so 'no items' is a data artifact that would leak the outcome.")

pop = o[o.in_population]
summ = []
for t in TARGETS:
    s = pop[t].dropna(); k, n = int(s.sum()), len(s); lo, hi = wilson(k, n)
    summ.append(dict(target=t, orders=n, events=k, rate_pct=round(k / n * 100, 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
summ = pd.DataFrame(summ)
print(f"\nResolved population: {len(pop):,} orders purchased up to {(OBS_END - pd.Timedelta(days=RESOLVE_DAYS)).date()}")
display(summ)
print("Overlap of the components inside 'went_wrong' (rows: late, columns: bad review; canceled shown separately):")
display(pd.crosstab(pop.late.map({0.0: "on time", 1.0: "late"}).fillna("canceled"), pop.bad_review.map({0: "review ok/none", 1: "bad review"}), margins=True))
print(f"Orders without any review: {(1 - pop.has_review.mean())*100:.1f}% (counted as 'not a bad review').")

Orders with NO item rows: 775, of which canceled/unavailable: 99.0%.
  -> excluded: at purchase time every order has items, so 'no items' is a data artifact that would leak the outcome.

Resolved population: 86,790 orders purchased up to 2018-07-05


,target,orders,events,rate_pct,ci_low,ci_high
0,went_wrong,86790,15375,17.72,17.46,17.97
1,late,86389,7525,8.71,8.52,8.90
2,bad_review,86790,12649,14.57,14.34,14.81
3,canceled,86790,401,0.46,0.42,0.51


Overlap of the components inside 'went_wrong' (rows: late, columns: bad review; canceled shown separately):


bad_review,bad review,review ok/none,All
late,,,
canceled,318,83,401
late,4882,2643,7525
on time,7449,71415,78864
All,12649,74141,86790


Orders without any review: 0.8% (counted as 'not a bad review').


In [8]:
it = items.merge(products[["product_id", "product_category_name", "product_weight_g", "product_length_cm",
                            "product_height_cm", "product_width_cm"]], on="product_id", how="left", validate="many_to_one")
it["vol_cm3"] = it.product_length_cm * it.product_height_cm * it.product_width_cm
if trans is not None:
    it = it.merge(trans, on="product_category_name", how="left", validate="many_to_one")
    it["category"] = it.product_category_name_english.fillna(it.product_category_name)
else:
    it["category"] = it.product_category_name
agg = it.groupby("order_id").agg(n_items=("order_item_id", "count"), n_products=("product_id", "nunique"),
                                 n_sellers=("seller_id", "nunique"), total_price=("price", "sum"), total_freight=("freight_value", "sum"),
                                 total_weight_g=("product_weight_g", lambda s: s.sum(min_count=len(s))),
                                 max_weight_g=("product_weight_g", "max"),
                                 total_volume_cm3=("vol_cm3", lambda s: s.sum(min_count=len(s)))).reset_index()
prim = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "seller_id", "category"]]
prim = prim.merge(sellers.rename(columns={"seller_zip_code_prefix": "seller_zip"})[["seller_id", "seller_zip", "seller_state"]],
                  on="seller_id", how="left", validate="many_to_one")
pt = pay.sort_values(["order_id", "payment_value"], ascending=[True, False]).groupby("order_id").agg(
    payment_type=("payment_type", "first"), installments=("payment_installments", "max")).reset_index()

o = attach(attach(attach(o, agg), prim), pt)
o = o.sort_values(["order_purchase_timestamp", "order_id"]).reset_index(drop=True)
o["customer_prior_orders"] = o.groupby("customer_unique_id").cumcount()      # orders placed earlier by the same person

# zip-prefix centroids (drop coordinates outside Brazil's bounding box)
g = geo[geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)]
zc = g.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
ZIP_CENTROIDS = {int(z): (float(a), float(b)) for z, (a, b) in zc.iterrows()}

# --- point-in-time seller history -------------------------------------------------------------
os_ = o[o.seller_id.notna()]
EVENTS = {
    "orders": os_.assign(known_at=os_.order_purchase_timestamp, value=1),
    "late":   os_[os_.late.notna()].assign(known_at=lambda d: d.order_delivered_customer_date.fillna(d.order_estimated_delivery_date), value=lambda d: d.late),
    "bad":    os_[os_.has_review == 1].assign(known_at=lambda d: d.review_answered, value=lambda d: d.bad_review),
    "cancel": os_.assign(known_at=os_.order_estimated_delivery_date, value=os_.canceled),
}
EVENTS = {k: v[["order_id", "seller_id", "known_at", "value"]].dropna(subset=["known_at"]).sort_values("known_at") for k, v in EVENTS.items()}

def prior_counts(ev, query, ts_col, prefix):
    """For each query row: number (n) and sum (k) of the seller's events known STRICTLY before query[ts_col]."""
    e = ev.copy(); e["known_at"] = pd.to_datetime(e["known_at"]).astype("datetime64[ns]")
    e["_n"] = e.groupby("seller_id").cumcount() + 1
    e["_k"] = e.groupby("seller_id")["value"].cumsum()
    q = query[["order_id", "seller_id", ts_col]].copy(); q[ts_col] = pd.to_datetime(q[ts_col]).astype("datetime64[ns]")
    q = q.sort_values(ts_col)
    m = pd.merge_asof(q, e[["seller_id", "known_at", "_n", "_k"]], left_on=ts_col, right_on="known_at",
                      by="seller_id", allow_exact_matches=False, direction="backward")
    return m[["order_id", "_n", "_k"]].rename(columns={"_n": f"{prefix}_n", "_k": f"{prefix}_k"}).fillna(0)

hist = os_[["order_id"]]
for name, prefix in [("orders", "seller_orders"), ("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
    hist = hist.merge(prior_counts(EVENTS[name], os_, "order_purchase_timestamp", prefix), on="order_id", how="left", validate="one_to_one")
hist = hist.rename(columns={"seller_orders_n": "seller_prior_orders"}).drop(columns="seller_orders_k")
o = attach(o, hist)

GLOBAL_RATES = {k: float(EVENTS[e].loc[EVENTS[e].known_at < TRAIN_END_GAPPED, "value"].mean())
                for k, e in [("late", "late"), ("bad_review", "bad"), ("cancel", "cancel")]}
print("Global rates used for shrinkage (training period only):", {k: round(v, 4) for k, v in GLOBAL_RATES.items()})

Global rates used for shrinkage (training period only): {'late': 0.0436, 'bad_review': 0.1311, 'cancel': 0.0128}


In [9]:
rng = np.random.default_rng(SEED)
sample = o[o.seller_id.notna()].iloc[rng.choice(int(o.seller_id.notna().sum()), 300, replace=False)]
bad_rows = 0
for _, r in sample.iterrows():
    for name, prefix in [("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
        ev = EVENTS[name]; s = ev[(ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)]
        if len(s) != r[f"{prefix}_n"] or s.value.sum() != r[f"{prefix}_k"]:
            bad_rows += 1
    ev = EVENTS["orders"]
    if ((ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)).sum() != r.seller_prior_orders:
        bad_rows += 1
    if r.order_id in set(EVENTS["late"].loc[(EVENTS["late"].seller_id == r.seller_id) & (EVENTS["late"].known_at < r.order_purchase_timestamp), "order_id"]):
        bad_rows += 1   # an order must never see its own outcome
assert bad_rows == 0, f"{bad_rows} mismatches - seller history is not point-in-time"
print("Leakage check passed: 300/300 orders match a brute-force recomputation, and no order sees its own outcome.")

Leakage check passed: 300/300 orders match a brute-force recomputation, and no order sees its own outcome.


In [10]:
# order_features.py - ONE feature function shared by the v3 notebook (training) and the app (scoring).
# The notebook embeds this exact file as a cell, so training and serving cannot drift apart.
import math
import numpy as np
import pandas as pd

NUM = ["n_items", "n_products", "n_sellers", "log_price", "log_freight", "freight_share",
       "total_weight_kg", "max_weight_kg", "total_volume_l", "dims_missing",
       "distance_km", "same_state", "promised_days", "installments",
       "purchase_hour", "purchase_dow", "is_holiday_season",
       "seller_prior_orders_log", "seller_has_history", "seller_prior_late_rate",
       "seller_prior_bad_review_rate", "seller_prior_cancel_rate", "customer_prior_orders"]
CAT = ["customer_state", "seller_state", "category", "payment_type"]
FEATURES = NUM + CAT
SHRINK_A = 20.0   # pseudo-orders used to shrink a seller's rates toward the global rate


def haversine_km(lat1, lng1, lat2, lng2):
    if any(v is None or (isinstance(v, float) and math.isnan(v)) for v in (lat1, lng1, lat2, lng2)):
        return float("nan")
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(a))


def _num(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return float("nan")
    return x


def shrunk_rate(k, n, prior, a=SHRINK_A):
    k, n = _num(k), _num(n)
    k = 0.0 if math.isnan(k) else k
    n = 0.0 if math.isnan(n) else n
    return (k + a * prior) / (n + a)


def features_from_inputs(o, zip_centroids, global_rates):
    """o: dict of raw order inputs known at purchase time (see keys below).
    zip_centroids: {zip_prefix(int): (lat, lng)}. global_rates: {"late":..,"bad_review":..,"cancel":..}.
    Seller history (seller_prior_*) must be computed as of the purchase moment by the caller."""
    def zc(z):
        try:
            return zip_centroids.get(int(z), (float("nan"), float("nan")))
        except (TypeError, ValueError):
            return (float("nan"), float("nan"))

    clat, clng = zc(o.get("customer_zip"))
    slat, slng = zc(o.get("seller_zip"))
    ts = pd.Timestamp(o["purchase_ts"])
    est = o.get("estimated_delivery_date")
    promised = (pd.Timestamp(est) - ts).total_seconds() / 86400 if est is not None and not pd.isna(est) else float("nan")
    price, freight = _num(o.get("total_price")), _num(o.get("total_freight"))
    den = price + freight
    vol = _num(o.get("total_volume_cm3"))
    n_prior = _num(o.get("seller_prior_orders")); n_prior = 0.0 if math.isnan(n_prior) else n_prior
    cs, ss = str(o.get("customer_state") or "missing"), str(o.get("seller_state") or "missing")
    f = {
        "n_items": min(_num(o.get("n_items")), 10),
        "n_products": min(_num(o.get("n_products")), 10),
        "n_sellers": min(_num(o.get("n_sellers")), 5),
        "log_price": math.log1p(max(price, 0)) if not math.isnan(price) else float("nan"),
        "log_freight": math.log1p(max(freight, 0)) if not math.isnan(freight) else float("nan"),
        "freight_share": freight / den if den and den > 0 else float("nan"),
        "total_weight_kg": _num(o.get("total_weight_g")) / 1000,
        "max_weight_kg": _num(o.get("max_weight_g")) / 1000,
        "total_volume_l": vol / 1000,
        "dims_missing": float(math.isnan(vol)),
        "distance_km": min(haversine_km(clat, clng, slat, slng), 4500),
        "same_state": float(cs == ss and cs != "missing"),
        "promised_days": min(max(promised, 0), 90) if not math.isnan(promised) else float("nan"),
        "installments": min(_num(o.get("installments")), 24),
        "purchase_hour": float(ts.hour),
        "purchase_dow": float(ts.dayofweek),
        "is_holiday_season": float(ts.month in (11, 12)),
        "seller_prior_orders_log": math.log1p(n_prior),
        "seller_has_history": float(n_prior >= 5),
        "seller_prior_late_rate": shrunk_rate(o.get("seller_late_k"), o.get("seller_late_n"), global_rates["late"]),
        "seller_prior_bad_review_rate": shrunk_rate(o.get("seller_bad_k"), o.get("seller_bad_n"), global_rates["bad_review"]),
        "seller_prior_cancel_rate": shrunk_rate(o.get("seller_cancel_k"), o.get("seller_cancel_n"), global_rates["cancel"]),
        "customer_prior_orders": min(_num(o.get("customer_prior_orders")) if not math.isnan(_num(o.get("customer_prior_orders"))) else 0.0, 10),
        "customer_state": cs, "seller_state": ss,
        "category": str(o.get("category") or "missing"),
        "payment_type": str(o.get("payment_type") or "missing"),
    }
    return f


def build_frame(rows, zip_centroids, global_rates):
    """Apply features_from_inputs to many orders (list of dicts or a DataFrame) -> DataFrame[FEATURES]."""
    if isinstance(rows, pd.DataFrame):
        rows = rows.to_dict("records")
    return pd.DataFrame([features_from_inputs(r, zip_centroids, global_rates) for r in rows], columns=FEATURES)

In [11]:
raw = o.rename(columns={"customer_zip_code_prefix": "customer_zip", "order_purchase_timestamp": "purchase_ts",
                        "order_estimated_delivery_date": "estimated_delivery_date"})
INPUT_COLS = ["customer_zip", "customer_state", "seller_zip", "seller_state", "purchase_ts", "estimated_delivery_date",
              "n_items", "n_products", "n_sellers", "total_price", "total_freight", "total_weight_g", "max_weight_g", "total_volume_cm3",
              "category", "payment_type", "installments", "seller_prior_orders", "seller_late_k", "seller_late_n",
              "seller_bad_k", "seller_bad_n", "seller_cancel_k", "seller_cancel_n", "customer_prior_orders"]
X_all = build_frame(raw[INPUT_COLS], ZIP_CENTROIDS, GLOBAL_RATES)
df = pd.concat([o[["order_id", "customer_unique_id", "seller_id", "order_purchase_timestamp", "order_status", "in_population"] + TARGETS].reset_index(drop=True),
                X_all.reset_index(drop=True)], axis=1)
assert len(df) == len(o) and df.order_id.is_unique
print(f"Feature frame: {len(df):,} orders x {len(FEATURES)} features | missing distance: {df.distance_km.isna().mean()*100:.1f}%")
display(df.loc[df.in_population, NUM].describe().T[["mean", "50%", "min", "max"]].round(2))

Feature frame: 99,441 orders x 27 features | missing distance: 1.3%


,mean,50%,min,max
n_items,1.14,1.00,1.00,10.00
n_products,1.04,1.00,1.00,7.00
n_sellers,1.01,1.00,1.00,5.00
log_price,4.46,4.48,0.62,9.51
log_freight,2.98,2.89,0.00,7.49
freight_share,0.21,0.18,0.00,0.96
total_weight_kg,2.45,0.80,0.00,184.40
max_weight_kg,2.17,0.70,0.00,40.42
total_volume_l,17.80,7.60,0.17,1476.00
dims_missing,0.00,0.00,0.00,1.00


## 2. Population, outcome and the 43 v2.1 features (identical to v2.1)

In [12]:
vo = o[o.is_value_order].sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"]).copy()
vo["days_since_first"] = (vo.order_purchase_timestamp - vo.groupby("customer_unique_id").order_purchase_timestamp.transform("min")).dt.total_seconds() / 86400
fo = vo.drop_duplicates("customer_unique_id").copy()

rv_mean = reviews.groupby("order_id").agg(first_order_csat=("review_score", "mean"),
                                         review_created=("review_creation_date", "min"),
                                         comment=("review_comment_message", lambda s: s.dropna().astype(str).str.len().max() if s.notna().any() else np.nan)).reset_index()
fo = attach(fo, rv_mean)
both = fo.order_delivered_customer_date.notna() & fo.review_answered.notna()
fo["scoring_ts"] = fo[["order_delivered_customer_date", "review_answered"]].max(axis=1).where(both)

occ = vo.loc[vo.days_since_first >= MIN_GAP_DAYS, ["customer_unique_id", "order_purchase_timestamp"]].rename(columns={"order_purchase_timestamp": "occ_ts"})
occ = occ.merge(fo[["customer_unique_id", "scoring_ts"]], on="customer_unique_id", how="left", validate="many_to_one")
occ["delta"] = (occ.occ_ts - occ.scoring_ts).dt.total_seconds() / 86400
before = occ[occ.delta <= 0].groupby("customer_unique_id").size().rename("n_before").reset_index()
after = occ[occ.delta > 0].groupby("customer_unique_id").delta.min().rename("days_to_repeat_after_score").reset_index()
fo = fo.merge(before, on="customer_unique_id", how="left", validate="one_to_one").merge(after, on="customer_unique_id", how="left", validate="one_to_one")
fo["at_risk"] = fo.scoring_ts.notna() & fo.n_before.isna()
for W in WINDOWS:
    fo[f"eligible_{W}"] = fo.at_risk & (fo.scoring_ts <= OBS_END - pd.Timedelta(days=W))
    fo[f"repeat_{W}"] = fo.days_to_repeat_after_score.le(W).astype(int)
assert fo.customer_unique_id.is_unique

rows = []
for W in WINDOWS:
    e = fo[fo[f"eligible_{W}"]]
    gap = e[e.scoring_ts <= TEST_START - pd.Timedelta(days=W)]
    rows.append(dict(window_days=W, eligible=len(e), events=int(e[f"repeat_{W}"].sum()), rate_pct=round(e[f"repeat_{W}"].mean() * 100, 2),
                     gapped_train_customers=len(gap), gapped_train_events=int(gap[f"repeat_{W}"].sum())))
pops = pd.DataFrame(rows)
print("v2 real-data reference for 180 days: 51,875 eligible / 902 events; gapped training set 4,037 customers / 65 events")
display(pops)

v2 real-data reference for 180 days: 51,875 eligible / 902 events; gapped training set 4,037 customers / 65 events


,window_days,eligible,events,rate_pct,gapped_train_customers,gapped_train_events
0,180,51876,902,1.74,4038,65
1,90,72304,739,1.02,12382,138


In [13]:
# v2's original 8 features, computed exactly as in v2
it2 = items.groupby("order_id").agg(f_items=("order_item_id", "count"), f_uprod=("product_id", "nunique"),
                                    f_items_value=("price", "sum"), f_freight=("freight_value", "sum")).reset_index()
sp2 = pay.groupby("order_id").agg(f_spend=("payment_value", "sum"), n_payment_methods=("payment_type", "nunique"),
                                  paid_voucher=("payment_type", lambda s: float((s == "voucher").any()))).reset_index()
fo = attach(attach(fo, it2), sp2)
den = fo.f_items_value + fo.f_freight
fo["first_order_items"] = fo.f_items.clip(1, 10)
fo["multi_product"] = (fo.f_uprod >= 2).astype(float).where(fo.f_uprod.notna())
fo["log_spend"] = np.log1p(fo.f_spend)
fo["freight_share"] = (fo.f_freight / den).where(den > 0).clip(0, 1)
fo["transit_days"] = ((fo.order_delivered_customer_date - fo.order_purchase_timestamp).dt.total_seconds() / 86400).clip(0, 60)
delay = (fo.order_delivered_customer_date - fo.order_estimated_delivery_date).dt.total_seconds() / 86400
fo["is_delivery_late"] = (delay > 0).astype(float).where(delay.notna())
fo["is_sp"] = (fo.customer_state == "SP").astype(float)
BASE = ["first_order_items", "multi_product", "log_spend", "freight_share", "first_order_csat", "transit_days", "is_delivery_late", "is_sp"]

# Extra experience detail (known at scoring time: after delivery and the review)
fo["delay_vs_estimate_days"] = delay.clip(-40, 60)
fo["has_review_comment"] = fo.comment.notna().astype(float)
fo["comment_len_log"] = np.log1p(fo.comment.fillna(0))
fo["review_answer_hours_log"] = np.log1p(((fo.review_answered - fo.review_created).dt.total_seconds() / 3600).clip(0, 24 * 60))
fo["days_to_score"] = ((fo.scoring_ts - fo.order_purchase_timestamp).dt.total_seconds() / 86400).clip(0, 120)

# Product listing of the main item
lst = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "product_id"]] \
        .merge(products[["product_id", "product_photos_qty", "product_description_lenght"]], on="product_id", how="left", validate="many_to_one")
lst = lst.rename(columns={"product_photos_qty": "photos_qty"}).assign(desc_len_log=lambda d: np.log1p(d.product_description_lenght))[["order_id", "photos_qty", "desc_len_log"]]
fo = attach(fo, lst)

# Purchase-time features from the shared v3 function (seller track record as of the purchase, distance, promise, category...)
pt = df[["order_id"] + FEATURES].rename(columns={c: "pt_" + c for c in FEATURES})
fo = attach(fo, pt)
CONSUMABLE = {"health_beauty", "perfumery", "pet_shop", "food", "food_drink", "drinks"}
fo["is_consumable"] = fo.pt_category.isin(CONSUMABLE).astype(float)

GROUPS = {
    "v2 original 8": BASE,
    "basket, price & payment": ["pt_n_items", "pt_n_products", "pt_n_sellers", "pt_log_price", "pt_log_freight", "pt_total_weight_kg",
                                "pt_max_weight_kg", "pt_total_volume_l", "pt_dims_missing", "pt_installments", "n_payment_methods",
                                "paid_voucher", "pt_payment_type"],
    "seller track record": ["pt_seller_prior_orders_log", "pt_seller_has_history", "pt_seller_prior_late_rate",
                            "pt_seller_prior_bad_review_rate", "pt_seller_prior_cancel_rate"],
    "geography & promise": ["pt_distance_km", "pt_same_state", "pt_promised_days", "pt_customer_state", "pt_seller_state"],
    "category & listing": ["pt_category", "is_consumable", "photos_qty", "desc_len_log"],
    "review detail & timing": ["delay_vs_estimate_days", "has_review_comment", "comment_len_log", "review_answer_hours_log", "days_to_score"],
    "purchase timing": ["pt_purchase_hour", "pt_purchase_dow", "pt_is_holiday_season"],
}
CAT21 = ["pt_customer_state", "pt_seller_state", "pt_category", "pt_payment_type"]
ALL = [f for g in GROUPS.values() for f in g]
assert len(ALL) == len(set(ALL)), "duplicate feature names"
NUM21 = [f for f in ALL if f not in CAT21]
print(f"v2 used {len(BASE)} features; v2.1 uses {len(ALL)} ({len(NUM21)} numeric, {len(CAT21)} categorical) in {len(GROUPS)} groups.")
display(fo.loc[fo.eligible_180, NUM21].describe().T[["count", "mean", "50%", "min", "max"]].round(2))

v2 used 8 features; v2.1 uses 43 (39 numeric, 4 categorical) in 7 groups.


,count,mean,50%,min,max
first_order_items,51876.0,1.14,1.00,1.00,10.00
multi_product,51876.0,0.03,0.00,0.00,1.00
log_spend,51875.0,4.69,4.65,2.40,9.52
freight_share,51876.0,0.21,0.18,0.00,0.85
first_order_csat,51876.0,4.17,5.00,1.00,5.00
transit_days,51876.0,12.84,11.05,0.53,60.00
is_delivery_late,51876.0,0.06,0.00,0.00,1.00
is_sp,51876.0,0.41,0.00,0.00,1.00
pt_n_items,51876.0,1.14,1.00,1.00,10.00
pt_n_products,51876.0,1.04,1.00,1.00,7.00


## 3. Model helpers

In [14]:
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

def make_logit(num, cat, C=0.3):
    parts = [("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                               ("clip", FunctionTransformer(np.clip, kw_args={"a_min": -5, "a_max": 5}))]), num)]
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=100), cat))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", LogisticRegression(max_iter=3000, C=C, random_state=SEED))])

def make_hgb(num, cat):
    pre = ColumnTransformer([("num", "passthrough", num)] + ([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)] if cat else []))
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04, max_iter=200, l2_regularization=2.0,
                                                                          min_samples_leaf=200, categorical_features=list(range(len(num), len(num) + len(cat))) or None,
                                                                          random_state=SEED))])

CANDIDATES = {
    "v2 baseline (8 features, logistic)": lambda: make_logit(BASE, []),
    "v2.1 logistic (all features)":       lambda: make_logit(NUM21, CAT21),
    "v2.1 gradient boosting (all)":       lambda: make_hgb(NUM21, CAT21),
}
FEATS = {"v2 baseline (8 features, logistic)": BASE}

def boot_auc(y, p, n=300):
    r = np.random.default_rng(SEED); idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = r.choice(idx, len(idx))
        if y[s].min() != y[s].max():
            out.append(roc_auc_score(y[s], p[s]))
    return [float(x) for x in np.percentile(out, [2.5, 97.5])]

def bands(p_train, p_test, y):
    q50, q90 = np.quantile(p_train, [.5, .9]); b = np.where(p_test >= q90, "High", np.where(p_test >= q50, "Medium", "Low"))
    out = {}
    for name in ("High", "Medium", "Low"):
        yy = y[b == name]; k, n = int(yy.sum()), len(yy); out[name] = (round(100 * k / max(n, 1), 2), *[round(v, 2) for v in wilson(k, n)], n)
    return out


H, PERIODS = 180, [0, 30, 60, 90, 120, 150, 180]           # 6 periods of 30 days
TEST_CUSTOMERS = fo[fo.eligible_180 & (fo.scoring_ts >= TEST_START)].copy()
CUTOFF = TEST_START                                          # training may only use what was known on this date
TRAIN_UNIVERSE = fo[fo.at_risk & (fo.scoring_ts < CUTOFF)].copy()
TRAIN_UNIVERSE["follow"] = ((CUTOFF - TRAIN_UNIVERSE.scoring_ts).dt.total_seconds() / 86400).clip(upper=H)
TRAIN_UNIVERSE["t_event"] = TRAIN_UNIVERSE.days_to_repeat_after_score.where(
    TRAIN_UNIVERSE.days_to_repeat_after_score <= TRAIN_UNIVERSE.follow)           # a return only counts if it happened before the cut-off
print(f"Test customers: {len(TEST_CUSTOMERS):,} ({int(TEST_CUSTOMERS.repeat_180.sum())} returned within 180 days)")
print(f"Training universe (scored before {CUTOFF.date()}): {len(TRAIN_UNIVERSE):,} customers, "
      f"{int(TRAIN_UNIVERSE.t_event.notna().sum())} returns observed before the cut-off; "
      f"median follow-up {TRAIN_UNIVERSE.follow.median():.0f} days")
print(f"For comparison, v2.1's gapped training set: {int((fo.eligible_180 & (fo.scoring_ts <= TEST_START - pd.Timedelta(days=H))).sum()):,} customers")

Test customers: 28,494 (443 returned within 180 days)
Training universe (scored before 2017-10-01): 23,382 customers, 261 returns observed before the cut-off; median follow-up 96 days
For comparison, v2.1's gapped training set: 4,038 customers


## 4. Category & product return rates (point-in-time, shrunk, out-of-fold)
Target for the rates: returned within the follow-up observed before the cut-off (customers with < 30 days of follow-up are
left out of the rate calculation). Training customers get **out-of-fold** rates, so a customer's own return never feeds
their own feature. Test customers get rates computed from the whole training universe.

In [15]:
from sklearn.model_selection import KFold

def rate_table(d, key, a):
    d = d[d.follow >= 30]
    g = d.groupby(key).agg(n=("t_event", "size"), k=("t_event", lambda s: s.notna().sum()))
    prior = d.t_event.notna().mean()
    return ((g.k + a * prior) / (g.n + a)).to_dict(), float(prior)

def encode(train, target_frame, key, a):
    table, prior = rate_table(train, key, a)
    return target_frame[key].map(table).fillna(prior).astype(float)

prim_prod = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "product_id"]] \
              .rename(columns={"product_id": "main_product_id"})
for d in ("fo", "TRAIN_UNIVERSE", "TEST_CUSTOMERS"):
    globals()[d] = globals()[d].merge(prim_prod, on="order_id", how="left", validate="many_to_one")
for d in (fo, TRAIN_UNIVERSE, TEST_CUSTOMERS):
    d["pt_category"] = d.pt_category.fillna("missing"); d["main_product_id"] = d.main_product_id.fillna("missing")

ENC = {"category_return_rate": ("pt_category", 30), "product_return_rate": ("main_product_id", 20)}
def add_encodings(train_universe, others):
    """OOF encodings for the training universe, full-universe encodings for the other frames."""
    tu = train_universe.copy()
    for name, (key, a) in ENC.items():
        tu[name] = np.nan
        for tr_idx, va_idx in KFold(5, shuffle=True, random_state=SEED).split(tu):
            tu.iloc[va_idx, tu.columns.get_loc(name)] = encode(tu.iloc[tr_idx], tu.iloc[va_idx], key, a).values
    outs = []
    for o_ in others:
        o_ = o_.copy()
        for name, (key, a) in ENC.items():
            o_[name] = encode(tu, o_, key, a)
        outs.append(o_)
    return tu, outs

TRAIN_UNIVERSE, (TEST_CUSTOMERS,) = add_encodings(TRAIN_UNIVERSE, [TEST_CUSTOMERS])
chk = TEST_CUSTOMERS.groupby(pd.qcut(TEST_CUSTOMERS.category_return_rate.rank(method="first"), 4, labels=["lowest", "low", "high", "highest"]), observed=True) \
                    .repeat_180.agg(["size", "mean"])
chk["mean"] = (chk["mean"] * 100).round(2)
print("Sanity check on the TEST period - 180-day return rate by quarter of the (pre-test) category return rate:")
display(chk.rename(columns={"size": "customers", "mean": "return_rate_pct"}))

Sanity check on the TEST period - 180-day return rate by quarter of the (pre-test) category return rate:


,customers,return_rate_pct
category_return_rate,,
lowest,7124,1.32
low,7123,1.29
high,7123,1.97
highest,7124,1.64


## 5. Review-text sentiment learned from Olist's own reviews (comment → stars), time-safe

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

txt = reviews.dropna(subset=["review_comment_message"]).copy()
txt["text"] = (txt.review_comment_title.fillna("") + " " + txt.review_comment_message.astype(str)).str.lower()
make_sent = lambda: Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=50000, strip_accents="unicode")),
                              ("ridge", Ridge(alpha=3.0))])
pre = txt[txt.review_answer_timestamp < CUTOFF]
first_txt = txt[txt.order_id.isin(fo.order_id)].groupby("order_id").text.apply(" ".join)

def sentiment_scores(fit_reviews, oof_orders):
    """Model fitted on fit_reviews; orders in oof_orders get out-of-fold scores, all other first orders full-model scores."""
    model = make_sent().fit(fit_reviews.text, fit_reviews.review_score)
    s = pd.Series(model.predict(first_txt.values), index=first_txt.index)
    own = fit_reviews[fit_reviews.order_id.isin(oof_orders)]
    if len(own):
        ids = np.array(sorted(own.order_id.unique()))
        for tr_i, va_i in KFold(5, shuffle=True, random_state=SEED).split(ids):
            va = set(ids[va_i]); m_ = make_sent().fit(fit_reviews[~fit_reviews.order_id.isin(va)].text, fit_reviews[~fit_reviews.order_id.isin(va)].review_score)
            idx = [o for o in first_txt.index if o in va]
            s.loc[idx] = m_.predict(first_txt.loc[idx].values)
    return s.clip(1, 5), model

sent, SENT_MODEL_PRE = sentiment_scores(pre, set(TRAIN_UNIVERSE.order_id))
for d in ("TRAIN_UNIVERSE", "TEST_CUSTOMERS"):
    x = globals()[d]; x["text_sentiment"] = x.order_id.map(sent); x["text_vs_stars"] = x.text_sentiment - x.first_order_csat
    globals()[d] = x
r = np.corrcoef(TEST_CUSTOMERS.dropna(subset=["text_sentiment"]).text_sentiment, TEST_CUSTOMERS.dropna(subset=["text_sentiment"]).first_order_csat)[0, 1]
print(f"Reviews with text before the cut-off: {len(pre):,}. First orders with a comment: {first_txt.size:,}. "
      f"On test customers, text sentiment vs stars correlation: {r:.2f}")

NEW = list(ENC) + ["text_sentiment", "text_vs_stars"]
ALL22 = ALL + NEW; NUM22 = NUM21 + NEW

Reviews with text before the cut-off: 10,824. First orders with a comment: 38,729. On test customers, text sentiment vs stars correlation: 0.82


## 6. The five candidates, all on the same test customers
* **v2.1 reference** — gradient boosting, 43 features, gapped training (should reproduce ≈ 0.557 on the real data)
* **A. more features** — v2.1 setup + the new features (isolates the feature gain)
* **B. survival** — survival model with v2.1's 43 features (isolates the more-data gain)
* **C. survival + more features**, gradient boosting
* **D. survival + more features**, logistic

In [17]:
def expand(d, feats):
    """Person-period rows: one row per 30-day period a customer was observed at risk; event = returned in that period."""
    parts = []
    for k in range(len(PERIODS) - 1):
        lo, hi = PERIODS[k], PERIODS[k + 1]
        at_risk = (d.follow > lo) & (d.t_event.isna() | (d.t_event > lo))
        ev = d.t_event.gt(lo) & d.t_event.le(hi)
        keep = at_risk & (ev | (d.follow >= hi))            # drop partially observed periods without an event
        p = d.loc[keep, feats].copy(); p["period"] = k; p["y"] = ev[keep].astype(int).values
        parts.append(p)
    return pd.concat(parts, ignore_index=True)

def hazard_predict(pipe, X, feats):
    surv = np.ones(len(X))
    for k in range(len(PERIODS) - 1):
        Xk = X[feats].copy(); Xk["period"] = k
        surv *= 1 - pipe.predict_proba(Xk)[:, 1]
    return 1 - surv                                          # P(return within 180 days)

def evaluate(name, p, p_train_like):
    y = TEST_CUSTOMERS.repeat_180.values
    dec = pd.DataFrame({"y": y, "p": p}); dec["d"] = 10 - pd.qcut(dec.p.rank(method="first"), 10, labels=False)
    return dict(model=name, oot_auc=float(roc_auc_score(y, p)), oot_ci=boot_auc(y, p),
                top_decile_lift=round(float(dec[dec.d == 1].y.mean() / y.mean()), 2), bands=bands(p_train_like, p, y))

RES22, PIPES = {}, {}
gap = fo[fo.eligible_180 & (fo.scoring_ts <= TEST_START - pd.Timedelta(days=H))].merge(
    TRAIN_UNIVERSE[["customer_unique_id"] + NEW], on="customer_unique_id", how="left", validate="one_to_one")

# reference and A: classic gapped training
for name, feats, num in [("v2.1 reference (GB, gapped)", ALL, NUM21), ("A. more features (GB, gapped)", ALL22, NUM22)]:
    pipe = make_hgb(num, CAT21).fit(gap[feats], gap.repeat_180)
    RES22[name] = evaluate(name, pipe.predict_proba(TEST_CUSTOMERS[feats])[:, 1], pipe.predict_proba(gap[feats])[:, 1]); PIPES[name] = (pipe, feats, "classifier")

# B, C, D: survival models trained on the whole training universe
for name, feats, num, maker in [("B. survival (GB, v2.1 features)", ALL, NUM21, "hgb"),
                                ("C. survival + more features (GB)", ALL22, NUM22, "hgb"),
                                ("D. survival + more features (logistic)", ALL22, NUM22, "logit")]:
    pp = expand(TRAIN_UNIVERSE, feats)
    pipe = (make_hgb(num + ["period"], CAT21) if maker == "hgb" else make_logit(num + ["period"], CAT21)).fit(pp[feats + ["period"]], pp.y)
    RES22[name] = evaluate(name, hazard_predict(pipe, TEST_CUSTOMERS, feats), hazard_predict(pipe, TRAIN_UNIVERSE, feats)); PIPES[name] = (pipe, feats, "hazard")
    print(f"{name}: trained on {len(pp):,} customer-periods from {len(TRAIN_UNIVERSE):,} customers ({int(pp.y.sum())} returns)")

ref = RES22["v2.1 reference (GB, gapped)"]["oot_auc"]
tbl = pd.DataFrame([{"model": r["model"], "oot_auc": round(r["oot_auc"], 3), "ci": f"{r['oot_ci'][0]:.3f}-{r['oot_ci'][1]:.3f}",
                     "top_decile_lift": r["top_decile_lift"], "High %": r["bands"]["High"][0], "High CI": f"{r['bands']['High'][1]}-{r['bands']['High'][2]}",
                     "Low %": r["bands"]["Low"][0], "Low CI": f"{r['bands']['Low'][1]}-{r['bands']['Low'][2]}",
                     "passes": (r["oot_ci"][0] > max(ref, 0.5)) if not r["model"].startswith("v2.1") else None} for r in RES22.values()])
print(f"\nTest: {len(TEST_CUSTOMERS):,} customers scored from {TEST_START.date()}, base rate {TEST_CUSTOMERS.repeat_180.mean()*100:.2f}%. "
      f"Rule: pass if CI low > v2.1 reference AUC ({ref:.3f}).")
display(tbl)
tbl.to_csv(OUT_DIR / "v2_2_results.csv", index=False)

B. survival (GB, v2.1 features): trained on 66,405 customer-periods from 23,382 customers (261 returns)
C. survival + more features (GB): trained on 66,405 customer-periods from 23,382 customers (261 returns)
D. survival + more features (logistic): trained on 66,405 customer-periods from 23,382 customers (261 returns)

Test: 28,494 customers scored from 2017-10-01, base rate 1.55%. Rule: pass if CI low > v2.1 reference AUC (0.557).


,model,oot_auc,ci,top_decile_lift,High %,High CI,Low %,Low CI,passes
0,"v2.1 reference (GB, gapped)",0.557,0.530-0.585,1.53,2.12,1.71-2.63,1.25,1.07-1.46,None
1,"A. more features (GB, gapped)",0.550,0.524-0.580,1.51,2.21,1.78-2.74,1.38,1.19-1.6,False
2,"B. survival (GB, v2.1 features)",0.545,0.520-0.575,1.13,1.77,1.46-2.15,1.29,1.1-1.52,False
3,C. survival + more features (GB),0.533,0.509-0.559,1.08,1.57,1.2-2.04,1.34,1.14-1.57,False
4,D. survival + more features (logistic),0.500,0.474-0.525,0.90,1.50,1.32-1.72,1.41,1.16-1.72,False


## 7. Decision and export (only if a candidate passes)

In [18]:
passing = tbl[tbl.passes == True].sort_values("oot_auc", ascending=False)
PROMOTED = None if passing.empty else passing.iloc[0].model
if PROMOTED is None:
    print("No v2.2 candidate passed. v2.1 stays the retention model; nothing exported.")
else:
    _, feats, kind = PIPES[PROMOTED]
    # Refit on everything known at the end of the data (follow-up censored at OBS_END), encodings & sentiment refitted too
    full = fo[fo.at_risk].copy()
    full["follow"] = ((OBS_END - full.scoring_ts).dt.total_seconds() / 86400).clip(upper=H)
    full["t_event"] = full.days_to_repeat_after_score.where(full.days_to_repeat_after_score <= full.follow)
    full, _ = add_encodings(full, [])
    sent_all, SENT_MODEL = sentiment_scores(txt, set(full.order_id))
    full["text_sentiment"] = full.order_id.map(sent_all); full["text_vs_stars"] = full.text_sentiment - full.first_order_csat
    num = [f for f in feats if f not in CAT21]
    if kind == "hazard":
        pp = expand(full, feats)
        final = (make_logit if "logistic" in PROMOTED else make_hgb)(num + ["period"], CAT21).fit(pp[feats + ["period"]], pp.y)
        p_fit = hazard_predict(final, full, feats)
    else:
        el = full[full.eligible_180]
        final = make_hgb(num, CAT21).fit(el[feats], el.repeat_180); p_fit = final.predict_proba(full[feats])[:, 1]
    enc_tables = {name: dict(zip(["table", "prior"], rate_table(full, key, a))) | {"key": key, "shrink": a} for name, (key, a) in ENC.items()}
    r = RES22[PROMOTED]
    art = {"model_version": "olist-retention-v2.2-" + pd.Timestamp.now().strftime("%Y%m%d"), "kind": "retention", "model_type": kind,
           "pipeline": final, "features": feats, "num": num, "cat": CAT21, "periods": PERIODS, "horizon_days": H,
           "encodings": enc_tables, "sentiment_model": SENT_MODEL if "text_sentiment" in feats else None,
           "sklearn_version": sklearn.__version__, "base_rate": float(TEST_CUSTOMERS.repeat_180.mean()),
           "score_quantiles": {"p50": float(np.quantile(p_fit, .5)), "p90": float(np.quantile(p_fit, .9))},
           "metrics": {"oot_auc": r["oot_auc"], "oot_auc_ci": r["oot_ci"], "top_decile_lift": r["top_decile_lift"],
                       "band_outcomes_oot": r["bands"], "reference_v2_1_auc": ref, "all_candidates": tbl.to_dict("records")},
           "scoring_time": "after delivery is complete and the first-order review has been answered",
           "population": "customers not yet returned at scoring time; outcome = returning order within 180 days after scoring"}
    joblib.dump(art, OUT_DIR / "retention_model_v2_2.joblib")
    out = full[["customer_unique_id", "order_id", "order_purchase_timestamp", "scoring_ts"] + feats].copy(); out["p_repeat"] = p_fit
    out.to_csv(OUT_DIR / "customers_scored_v2_2.csv", index=False)
    chk = joblib.load(OUT_DIR / "retention_model_v2_2.joblib")
    again = hazard_predict(chk["pipeline"], full.head(300), feats) if kind == "hazard" else chk["pipeline"].predict_proba(full.head(300)[feats])[:, 1]
    assert np.allclose(again, p_fit[:300]), "reload mismatch"
    print(f"PROMOTED: {PROMOTED}. Saved retention_model_v2_2.joblib and customers_scored_v2_2.csv ({len(out):,} customers), reload OK.")

No v2.2 candidate passed. v2.1 stays the retention model; nothing exported.


## 8. Auto-generated summary

In [19]:
r0 = RES22["v2.1 reference (GB, gapped)"]
print(f"v2.1 reference on this test: AUC {r0['oot_auc']:.3f} ({r0['oot_ci'][0]:.3f}-{r0['oot_ci'][1]:.3f}).")
for name in ["A. more features (GB, gapped)", "B. survival (GB, v2.1 features)", "C. survival + more features (GB)", "D. survival + more features (logistic)"]:
    r = RES22[name]
    print(f"{name:<40} AUC {r['oot_auc']:.3f} ({r['oot_ci'][0]:.3f}-{r['oot_ci'][1]:.3f}), change {r['oot_auc'] - r0['oot_auc']:+.3f}, "
          f"High {r['bands']['High'][0]}% vs Low {r['bands']['Low'][0]}%")
gain_data = RES22["B. survival (GB, v2.1 features)"]["oot_auc"] - r0["oot_auc"]
gain_feat = RES22["A. more features (GB, gapped)"]["oot_auc"] - r0["oot_auc"]
print(f"More data (survival) changed AUC by {gain_data:+.3f}; more features by {gain_feat:+.3f}.")
print("Verdict:", f"promote {PROMOTED}." if PROMOTED else "nothing passed - v2.1 stays. This is likely close to the ceiling for Olist; "
      "further retention gains need real customer history from the live loop.")

v2.1 reference on this test: AUC 0.557 (0.530-0.585).
A. more features (GB, gapped)            AUC 0.550 (0.524-0.580), change -0.007, High 2.21% vs Low 1.38%
B. survival (GB, v2.1 features)          AUC 0.545 (0.520-0.575), change -0.012, High 1.77% vs Low 1.29%
C. survival + more features (GB)         AUC 0.533 (0.509-0.559), change -0.024, High 1.57% vs Low 1.34%
D. survival + more features (logistic)   AUC 0.500 (0.474-0.525), change -0.057, High 1.5% vs Low 1.41%
More data (survival) changed AUC by -0.012; more features by -0.007.
Verdict: nothing passed - v2.1 stays. This is likely close to the ceiling for Olist; further retention gains need real customer history from the live loop.
